In [5]:
import numpy as np
import rasterio
from pathlib import Path
import os

# One-Hot Encode Land Cover Rasters

Converts single-band categorical land cover rasters (values 0–6, representing 7 land types from the Global LULC dataset) into 7-band binary one-hot encoded GeoTIFFs. Processes both 0.10° and 0.25° files for historical (2020) and all SSP scenarios.

**Input**: `READY_data/clipped_landuse_data/*.tif` (1-band categorical, from clip_resize_data.ipynb)
**Output**: `READY_data/landuse_onehot/*_onehot.tif` (7-band binary uint8)

In [ ]:
landuse_folder = "READY_data/clipped_landuse_data"
landuse_files = list(Path(landuse_folder).glob("*.tif"))

print("Checking landcover categories\n")
print("=" * 70)

# Collect all unique values across all files
all_categories = set()

for file in landuse_files[:3]:  # Check first 3 files
    with rasterio.open(file) as src:
        data = src.read(1)
        unique_vals = np.unique(data[~np.isnan(data)])
        all_categories.update(unique_vals.astype(int))
        
        print(f"{file.name}")
        print(f"  Unique categories: {sorted(unique_vals.astype(int))}")
        print(f"  Number of categories: {len(unique_vals)}")
        print()

print("=" * 70)
print(f"\nAll unique categories found: {sorted(all_categories)}")
print(f"Total number of categories: {len(all_categories)}")

print("\n" + "=" * 70)
print("ONE-HOT ENCODING OPTIONS:")
print("\n1. **Full one-hot encoding**: Create separate channel for each category")
print(f"   Result: {len(all_categories)} channels instead of 1")
print("   Pro: CNN can learn category relationships")
print("   Con: Large memory usage")
print("\n2. **Normalized categorical**: Keep as single channel, normalize to [0, 1]")
print(f"   Result: Single channel with values 0 to 1")
print("   Pro: Memory efficient")
print("   Con: Assumes ordinal relationship between categories")
print("\n3. **Grouped encoding**: Group similar categories, then one-hot")
print("   Pro: Reduces channels while preserving meaning")
print("   Con: Need to define groups")

In [ ]:
landuse_folder = "READY_data/clipped_landuse_data"
landuse_025_folder = "READY_data/clipped_landuse_data_025"
output_folder = "READY_data/landuse_onehot"
output_025_folder = "READY_data/landuse_onehot_025"

os.makedirs(output_folder, exist_ok=True)
os.makedirs(output_025_folder, exist_ok=True)

num_categories = 7
categories = [0, 1, 2, 3, 4, 5, 6]

print("One-hot encoding landuse files\n")
print("=" * 70)

# Process 0.1° files
print("\n### PROCESSING 0.1° FILES ###\n")
landuse_files = list(Path(landuse_folder).glob("*.tif"))

for i, file in enumerate(landuse_files, 1):
    print(f"[{i}/{len(landuse_files)}] Processing: {file.name}")
    
    output_file = os.path.join(output_folder, file.name.replace('.tif', '_onehot.tif'))
    
    with rasterio.open(file) as src:
        data = src.read(1)
        height, width = data.shape
        
        # Create one-hot encoded array: (num_categories, height, width)
        onehot = np.zeros((num_categories, height, width), dtype=np.uint8)
        
        # Fill one-hot channels
        for cat_idx, cat_value in enumerate(categories):
            onehot[cat_idx] = (data == cat_value).astype(np.uint8)
        
        # Update metadata for multi-band output
        out_meta = src.meta.copy()
        out_meta.update({
            'count': num_categories,
            'dtype': 'uint8',
            'compress': 'lzw'
        })
        
        # Write output
        with rasterio.open(output_file, 'w', **out_meta) as dst:
            dst.write(onehot)
        
        print(f"  ✓ Created: {num_categories} channels ({height} x {width})")

# Process 0.25° files
print("\n### PROCESSING 0.25° FILES ###\n")
landuse_025_files = list(Path(landuse_025_folder).glob("*.tif"))

for i, file in enumerate(landuse_025_files, 1):
    print(f"[{i}/{len(landuse_025_files)}] Processing: {file.name}")
    
    output_file = os.path.join(output_025_folder, file.name.replace('.tif', '_onehot.tif'))
    
    with rasterio.open(file) as src:
        data = src.read(1)
        height, width = data.shape
        
        # Create one-hot encoded array
        onehot = np.zeros((num_categories, height, width), dtype=np.uint8)
        
        # Fill one-hot channels
        for cat_idx, cat_value in enumerate(categories):
            onehot[cat_idx] = (data == cat_value).astype(np.uint8)
        
        # Update metadata
        out_meta = src.meta.copy()
        out_meta.update({
            'count': num_categories,
            'dtype': 'uint8',
            'compress': 'lzw'
        })
        
        # Write output
        with rasterio.open(output_file, 'w', **out_meta) as dst:
            dst.write(onehot)
        
        print(f"  ✓ Created: {num_categories} channels ({height} x {width})")

print("\n" + "=" * 70)
print("✓ One-hot encoding complete!")
print(f"\n0.1° files saved to: {output_folder}")
print(f"0.25° files saved to: {output_025_folder}")
print(f"\nEach file now has {num_categories} channels (one per category)")
print("Channel 0 = category 0, Channel 1 = category 1, etc.")